# Telecom X - Parte 2

## 1.0 Importación de Librerías

In [15]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
#from sklearn.inspection import permutation_importance

## 2.0 Preparación de los Datos

### 2.1 Extracción del Archivo Tratado

In [16]:
datos = pd.read_csv('/content/datos_tratados.csv')

In [17]:
datos.head()

,Customer_ID,Churn,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges_Monthly,Charges_Total,daily_accounts
0,0002-ORFBO,No,Female,0,1,1,9,1,0,DSL,...,0,1,1,0,One year,1,Mailed check,65.6,593.30,2.186667
1,0003-MKNFE,No,Male,0,0,0,9,1,1,DSL,...,0,0,0,1,Month-to-month,0,Mailed check,59.9,542.40,1.996667
2,0004-TLHLJ,Yes,Male,0,0,0,4,1,0,Fiber optic,...,1,0,0,0,Month-to-month,1,Electronic check,73.9,280.85,2.463333
3,0011-IGKFF,Yes,Male,1,1,0,13,1,0,Fiber optic,...,1,0,1,1,Month-to-month,1,Electronic check,98.0,1237.85,3.266667
4,0013-EXCHZ,Yes,Female,1,1,0,3,1,0,Fiber optic,...,0,1,1,0,Month-to-month,1,Mailed check,83.9,267.40,2.796667


In [18]:
datos.columns

Index(['Customer_ID', 'Churn', 'Gender', 'SeniorCitizen', 'Partner',
       'Dependents', 'Tenure', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'Charges_Monthly', 'Charges_Total',
       'daily_accounts'],
      dtype='object')

### 2.2 Eliminación de Columnas Irrelevantes

In [19]:
# Eliminar columna de Identificador único
datos = datos.drop(columns= 'Customer_ID')

In [20]:
# Verificar correlación entre posibles columnas redundantes
datos[['Charges_Monthly', 'Charges_Total', 'Tenure', 'daily_accounts']].corr()

,Charges_Monthly,Charges_Total,Tenure,daily_accounts
Charges_Monthly,1.000000,0.652109,0.247982,1.000000
Charges_Total,0.652109,1.000000,0.825118,0.652109
Tenure,0.247982,0.825118,1.000000,0.247982
daily_accounts,1.000000,0.652109,0.247982,1.000000


In [21]:
# Eliminar columna redundante
datos = datos.drop(columns=['daily_accounts'])

In [22]:
#Verificar cuántos NaN hay en Churn
print(datos['Churn'].isna().sum())

224


In [23]:
datos = datos.dropna(subset=['Churn'])

### 2.3 Encoding

In [24]:
#  Convertir 'Churn' a binario (ajusta si los valores son otros)
datos['Churn'] = datos['Churn'].map({'Yes': 1, 'No': 0})

#  Codificar variables categóricas
for col in datos.select_dtypes(include='object').columns:
    datos[col] = LabelEncoder().fit_transform(datos[col])

#  Separar características y objetivo
X = datos.drop('Churn', axis=1)
y = datos['Churn']

#  Entrenar Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

RandomForestClassifier(random_state=42)

In [25]:
datos

,Churn,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges_Monthly,Charges_Total
0,0,0,0,1,1,9,1,0,0,0,1,0,1,1,0,1,1,3,65.60,593.30
1,0,1,0,0,0,9,1,1,0,0,0,0,0,0,1,0,0,3,59.90,542.40
2,1,1,0,0,0,4,1,0,1,0,0,1,0,0,0,0,1,2,73.90,280.85
3,1,1,1,1,0,13,1,0,1,0,1,1,0,1,1,0,1,2,98.00,1237.85
4,1,0,1,1,0,3,1,0,1,0,0,0,1,1,0,0,1,3,83.90,267.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7262,0,0,0,0,0,13,1,0,0,1,0,0,1,0,0,1,0,3,55.15,742.90
7263,1,1,0,1,0,22,1,1,1,0,0,0,0,0,1,0,1,2,85.10,1873.70
7264,0,1,0,0,0,2,1,0,0,0,1,0,0,0,0,0,1,3,50.30,92.75
7265,0,1,0,1,1,67,1,0,0,1,0,1,1,0,1,2,0,3,67.85,4627.65


### 2.4 Verificación de la Proporción de Cancelación (Churn)

### 2.5 Normalización